In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache \
    import VisualBehaviorNeuropixelsProjectCache

%matplotlib inline

In [2]:
# Update this to a valid directory in your filesystem. This is where the data will be stored.
cache_dir = './data/'
cache = VisualBehaviorNeuropixelsProjectCache.from_local_cache(cache_dir=cache_dir) #from_s3_cache(cache_dir=cache_dir)

# get the metadata tables
units_table = cache.get_unit_table()
channels_table = cache.get_channel_table()
probes_table = cache.get_probe_table()
behavior_sessions_table = cache.get_behavior_session_table()
ecephys_sessions_table = cache.get_ecephys_session_table()

c:\Users\matth\anaconda3\envs\AMP3-Spring-2026-Project\Lib\site-packages\allensdk\api\cloud_cache\cloud_cache.py:547: OutdatedManifestWarning: You are loading visual-behavior-neuropixels_project_manifest_v0.5.0.json. A more up to date version of the dataset -- visual-behavior-ophys_project_manifest_v1.1.0.json -- exists online. To see the changes between the two versions of the dataset, run
VisualBehaviorNeuropixelsProjectCache.compare_manifests('visual-behavior-neuropixels_project_manifest_v0.5.0.json', 'visual-behavior-ophys_project_manifest_v1.1.0.json')
To load another version of the dataset, run
VisualBehaviorNeuropixelsProjectCache.load_manifest('visual-behavior-ophys_project_manifest_v1.1.0.json')
  warnings.warn(msg, OutdatedManifestWarning)


In [3]:
session_id = 1065437523 #1064644573
session = cache.get_ecephys_session(
            ecephys_session_id=session_id)

c:\Users\matth\anaconda3\envs\AMP3-Spring-2026-Project\Lib\site-packages\hdmf\spec\namespace.py:772: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Load Relevant Data

In [4]:
# Get Neural Data
units = session.get_units()
channels = session.get_channels()
unit_channels = units.merge(channels, left_on='peak_channel_id', right_index=True)

#first let's sort our units by depth
unit_channels = unit_channels.sort_values('probe_vertical_position', ascending=False)

#now we'll filter them
good_unit_filter = ((unit_channels['snr']>1)&
                    (unit_channels['isi_violations']<1)&
                    (unit_channels['firing_rate']>0.1))
good_units = unit_channels.loc[good_unit_filter]


# Unit info
unit_indices = np.array(good_units.index)
spike_times = dict([(i,session.spike_times[i]) for i in unit_indices])
structures = dict(good_units.structure_acronym)

# Stimulus Data
stimulus_presentations = session.stimulus_presentations
# Active stimulus
active_stimulus_presentations = stimulus_presentations[stimulus_presentations["active"]]
# Active Stimulus: Time, Name, Is Change
onset_times = active_stimulus_presentations['start_time'].values
image_names = active_stimulus_presentations['image_name'].values
image_is_changes = active_stimulus_presentations['is_change'].values


# Behavioral Data
eye_tracking = session.eye_tracking
running_speed = session.running_speed
licks = session.licks

Pre Compute Bins

In [73]:
first_onset_time = onset_times[0]
last_onset_time = onset_times[-1]

bin_size = 0.025
buffer = 1.0
bin_start_times = np.arange(first_onset_time-buffer, last_onset_time+buffer, bin_size)
bin_end_times = np.arange(first_onset_time+bin_size-buffer, last_onset_time+bin_size+buffer, bin_size)

num_bins = len(bin_start_times)

Get Binned Spike Counts

In [74]:
num_units = len(spike_times)

Y = np.zeros((num_units, num_bins)) # num of neurons, time steps

# For each unit
for k, unit_idx in enumerate(unit_indices):
    # Select corresponding unit's spike times
    unit_spike_times = spike_times[unit_idx]

    # Find the correct indices of the unit spike times for each bin
    start_indices = np.searchsorted(unit_spike_times, bin_start_times)
    stop_indices = np.searchsorted(unit_spike_times, bin_end_times)

    # Get the counts for each bin
    counts = stop_indices - start_indices
    Y[k] = counts

In [75]:
num_inputs = 8 + 1 + 2 + 3 # images, omit, task, behavior

X = np.zeros((num_inputs, num_bins))

Get Images + Omissions

In [76]:
# Getting Images

unique_images = list(np.unique(image_names))

for onset_time,image_name in zip(onset_times, image_names):
    # get image id
    image_id = unique_images.index(image_name)

    # get image time index
    time_idx = np.searchsorted(bin_start_times, onset_time)-1

    X[image_id, time_idx] = 1.0

Get Task

In [ ]:
# idk

Get Behavior

In [77]:
# Running
run_start_indices = np.searchsorted(running_speed.timestamps, bin_start_times)
run_end_indices = np.searchsorted(running_speed.timestamps, bin_end_times)

for time_idx, (run_start_idx, run_end_idx) in enumerate(zip(run_start_indices, run_end_indices)):
    bin_values = running_speed.speed.values[run_start_idx:run_end_idx]
    mean_value = np.nanmean(bin_values[~np.isnan(bin_values)])
    X[9+2, time_idx] = mean_value


# Pupil
eye_start_indices = np.searchsorted(eye_tracking.timestamps, bin_start_times)
eye_end_indices = np.searchsorted(eye_tracking.timestamps, bin_end_times)

for time_idx, (eye_start_idx, eye_end_idx) in enumerate(zip(eye_start_indices, eye_end_indices)):
    bin_values = eye_tracking.pupil_area.values[eye_start_idx:eye_end_idx]
    bin_values = bin_values[~np.isnan(bin_values)]

    # have to include this in case there's not enough values in time frame
    while len(bin_values) < 1:
        eye_start_idx -= 1
        eye_end_idx += 1
        bin_values = eye_tracking.pupil_area.values[eye_start_idx:eye_end_idx]
        bin_values = bin_values[~np.isnan(bin_values)]

    mean_value = np.nanmean(bin_values)
    X[9+2+1, time_idx] = mean_value


# Licking
start_indices = np.searchsorted(licks.timestamps, bin_start_times)
stop_indices = np.searchsorted(licks.timestamps, bin_end_times)

# Get the counts for each bin
counts = stop_indices - start_indices
X[9+2+2] = 1.0*(counts > 0.0)

C:\Users\matth\AppData\Local\Temp\ipykernel_14560\239471520.py:7: RuntimeWarning: Mean of empty slice
  mean_value = np.nanmean(bin_values[~np.isnan(bin_values)])


In [ ]:
# Get Units in Brain region of interest
area_of_interest = 'VISp'
binned_spike_counts = []
area_units = good_units[good_units['structure_acronym']==area_of_interest]
for iu, unit in area_units.iterrows():
    unit_spike_times = spike_times[iu]
    counts = np.histogram(spike_times[iu], bins)[0]
    binned_spike_counts.append(counts)
binned_spike_counts = np.array(binned_spike_counts)


Do Decoding

In [9]:
# Map Structure Acronym to Brain Region
acronym2region = {
    "APN": "Midbrain",
    "CA1": "Hippo",
    "CA3": "Hippo",
    "DG": "Hippo",
    "Eth": "Thalamus",
    "HPF": "Hippo",
    "LP": "Thalamus",
    "MB": "Midbrain",
    "MGd": "Midbrain",
    "MGm": "Midbrain",
    "MGv": "Midbrain",
    "MRN": "Midbrain",
    "NB": "UNKNOWN", # idk
    "NOT": "Midbrain",
    "PIL": "Thalamus",
    "POL": "Thalamus",
    "POST": "Hippo",
    "ProS": "Hippo",
    "SUB": "Hippo",
    "TH": "Thalamus",
    "VISal": "VIS",
    "VISam": "VIS",
    "VISl": "VIS",
    "VISp": "VIS",
    "VISpm": "VIS",
    "VISpm": "VIS",
    "VISrl": "VIS",
    "root": "UNKNOWN", # idk

    "SGN": "Thalamus",
    "PoT": "Thalamus",
    "PP": "Thalamus",
    "RN": "Midbrain", # i think
    "LT": "Midbrain",
}

In [10]:
unique_structures = np.unique(list(structures.values()))

temp_structure_hist = dict([(struct, []) for struct in unique_structures])

for i,unit_idx in enumerate(unit_indices):
    struct = structures[unit_idx]
    temp_structure_hist[struct].append(hist[:,:,i])

min_units = 1

print("Structure and # of good units")
structure_hists = dict()
for key,value in temp_structure_hist.items():
    if len(value) < min_units:
        continue

    print(key, len(value))

    arr = np.stack(value, axis=2)
    structure_hists[key] = arr

Structure and # of good units
APN 111
CA1 172
CA3 45
DG 84
LP 24
LT 1
MB 15
MGm 12
MGv 1
MRN 85
NOT 17
PIL 65
POST 1
PP 49
PoT 55
ProS 89
RN 16
SGN 231
SUB 44
TH 136
VISal 89
VISam 107
VISl 90
VISp 71
VISpm 120
VISrl 73


In [11]:
unique_regions = np.unique(list(acronym2region.values()))

temp_region_hist = dict([(region, []) for region in unique_regions])

for i,unit_idx in enumerate(unit_indices):
    region = acronym2region[structures[unit_idx]]
    temp_region_hist[region].append(hist[:,:,i])

min_units = 1

print("Region and # of good units")
region_hists = dict()
for key,value in temp_region_hist.items():
    if len(value) < min_units:
        continue

    print(key, len(value))

    arr = np.stack(value, axis=2)
    region_hists[key] = arr

Region and # of good units
Hippo 435
Midbrain 258
Thalamus 560
VIS 550


In [12]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score